# Week 4 — From a CLI bot to a real web app (+ tools)

Last week you deployed a pod to Kubernetes. This week your chatbot grows a **face** and **hands**:

- a **face** — a Streamlit web UI a stranger could use (streaming answers, memory, a persona)
- **hands** — *tool calling*, so the bot can do exact math and even **run read-only `kubectl`** to answer questions about your own cluster

We build it in five parts:

1. **The LLM bits** — streaming, memory, system prompts (test them right here, before the UI)
2. **Streamlit, step by step** — four versions of `app.py`, each one small change bigger than the last
3. **Tools show-and-tell** — teach the bot to call functions (a math tool, then a **kubectl** tool)
4. **Put it together** — a Streamlit app whose bot can answer *"how many pods are running in our namespace?"*
5. **Ship it onto the cluster** — package `app.py` in a ConfigMap and run it as a pod on NRP

> You need your **NRP token** in a `.env` file (same as Week 1). The Streamlit steps write real `app.py` files you run in a terminal with `streamlit run app.py`.

## Setup

```bash
pip install streamlit openai python-dotenv
```

Your `.env` (never commit it) needs:

```
NRP_LLM_TOKEN=sk-...your token...
NRP_LLM_BASE_URL=https://ellm.nrp-nautilus.io/v1
```

In [ ]:
import os, json, subprocess
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(
    api_key=os.environ["NRP_LLM_TOKEN"],
    base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"),
)

# Our cohort's shared namespace (the one the mentor posted in #rehs-2026).
NAMESPACE = "rehs-2026-chatbot"

# Heads-up: NRP's chat models "think" before they answer. gemma replies to simple
# factual questions directly, but open-ended prompts make it (and gpt-oss, qwen3...)
# reason silently first. That has two consequences we'll meet below (A2 and A4).
print("client ready ->", client.base_url)

: 

---
## Part A — The LLM bits

Before we wrap anything in a UI, get these three ideas solid: **streaming**, **memory**, and the **system prompt**. You met them in Week 2/Week 1; here they are in their final form.

### A1. Streaming — words as they're typed

Passing `stream=True` gives you the answer chunk-by-chunk instead of all at once. Two gotchas that *will* bite you:

1. The **last chunk has no choices** (it only carries usage stats) → always `if not chunk.choices: continue`.
2. `delta.content` can be `None` → `or ""`.

In [ ]:
def stream_answer(messages, model="gemma"):
    # Yields the answer piece by piece. For simple questions gemma streams content
    # right away; for open-ended ones it may pause to "think" first (see A2/A4).
    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    for chunk in stream:
        if not chunk.choices:          # last chunk = usage only, no choices
            continue
        yield chunk.choices[0].delta.content or ""

print("Bot: ", end="")
for piece in stream_answer([{"role": "user", "content": "In two sentences, explain what a GPU does."}]):
    print(piece, end="", flush=True)
print()

### A2. The reasoning-model gotcha

Most NRP models "think" before answering — `gpt-oss` does it heavily. Those thinking tokens arrive on `delta.reasoning`, *not* `delta.content`, so with a naive content-only loop the screen looks frozen, then the answer appears all at once. (Even `gemma` does this on open-ended prompts — you'll see it in A4.)

Knowing this, you can show a **"🧠 thinking…"** indicator instead of a dead screen (we do exactly that in the Streamlit sidebar step).

In [ ]:
stream = client.chat.completions.create(
    model="gpt-oss",
    messages=[{"role": "user", "content": "What is 2+2? Answer in one word."}],
    max_tokens=500, stream=True,
)
reasoning, content = "", ""
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    reasoning += getattr(delta, "reasoning", None) or ""
    content  += delta.content or ""

print("thinking (hidden):", repr(reasoning))
print("answer           :", repr(content))

### A3. Memory + a system prompt

The model has **no memory** — it only knows what's in the `messages` list you send. So we keep a list, append every user turn and every bot turn, and lead with a **system** message that sets the bot's voice.

In [ ]:
history = [
    {"role": "system", "content": "You are a terse NRP support bot. Answer in one short sentence, no fluff."},
]

def ask(user_text, model="gemma"):
    history.append({"role": "user", "content": user_text})
    answer = "".join(stream_answer(history, model=model))
    history.append({"role": "assistant", "content": answer})
    return answer

print("Q1:", ask("What is a pod?"))
print("Q2:", ask("And how is that different from a deployment?"))  # 'that' only works if it remembers
print("\nturns in memory:", len(history))

### A4. Parameters — and a NRP gotcha: `max_tokens` counts *thinking* too

`temperature` controls randomness (0 = deterministic, ~1+ = creative). `max_tokens` caps length — **but on NRP the thinking tokens (A2) count against that cap.** Set it too low on an open-ended prompt and the model spends the whole budget thinking, so `content` comes back **`None`** with `finish_reason="length"`. Rule of thumb: on NRP, leave `max_tokens` off (or generous) for anything that requires reasoning.

In [ ]:
prompt = [{"role": "user", "content": "Give a one-line tagline for a supercomputer."}]

# Too small: gemma spends all 50 tokens thinking, so the answer is starved.
r = client.chat.completions.create(model="gemma", messages=prompt, max_tokens=50)
print("max_tokens=50 ->", repr(r.choices[0].message.content),
      "| finish:", r.choices[0].finish_reason)

# No cap: it finishes thinking, then answers. temperature sets the vibe.
for temp in [0.0, 1.3]:
    r = client.chat.completions.create(model="gemma", messages=prompt, temperature=temp)
    print(f"temp={temp:<4} ->", (r.choices[0].message.content or "").strip())

---
## Part B — Streamlit, step by step

[Streamlit](https://docs.streamlit.io/) turns a Python script into a web app: no HTML, no JavaScript. You write normal Python; it renders a page and re-runs your whole script top-to-bottom on every interaction. State that must survive a re-run lives in `st.session_state`.

Each cell below **writes a real `app.py`** to disk. After running a cell, open a terminal and:

```bash
streamlit run app.py
```

It opens `http://localhost:8501`. Leave it running — edit, save, and Streamlit offers to rerun. Stop it with `Ctrl+C`.

### B1. The skeleton — a chat UI that just echoes

`st.chat_input` gives you the input box; `st.chat_message` renders a bubble; `st.session_state.messages` is our memory across reruns. No LLM yet — it echoes, so you can see the shell working.

In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="NRP Helper", page_icon="🤖")
st.title("🤖 NRP Helper")

if "messages" not in st.session_state:
    st.session_state.messages = []

# Replay the conversation so far (reruns wipe the screen, not session_state).
for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask about NRP..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)

    answer = f"You said: {prompt}"        # placeholder — no LLM yet
    st.session_state.messages.append({"role": "assistant", "content": answer})
    st.chat_message("assistant").write(answer)

### B2. Wire in the LLM (no streaming yet)

Swap the echo for a real call. Load `.env`, build the client, send the whole `messages` list (that's the memory), print the reply.

In [ ]:
%%writefile app.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"))

st.set_page_config(page_title="NRP Helper", page_icon="🤖")
st.title("🤖 NRP Helper")

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "system", "content": "You are a helpful, concise NRP support assistant."},
    ]

for msg in st.session_state.messages:
    if msg["role"] != "system":            # don't show the system prompt
        st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask about NRP..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)

    response = client.chat.completions.create(
        model="gemma", messages=st.session_state.messages,
    )
    answer = response.choices[0].message.content
    st.session_state.messages.append({"role": "assistant", "content": answer})
    st.chat_message("assistant").write(answer)

### B3. Stream it — the moment it feels alive

`st.write_stream` takes a generator and renders each piece as it arrives. We reuse the exact streaming trick from Part A (guard empty `choices`, `or ""`). This is the single biggest UX upgrade in the whole app.

In [ ]:
%%writefile app.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"))

st.set_page_config(page_title="NRP Helper", page_icon="🤖")
st.title("🤖 NRP Helper")

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "system", "content": "You are a helpful, concise NRP support assistant."},
    ]

for msg in st.session_state.messages:
    if msg["role"] != "system":
        st.chat_message(msg["role"]).write(msg["content"])

def token_stream(messages):
    stream = client.chat.completions.create(model="gemma", messages=messages, stream=True)
    for chunk in stream:
        if not chunk.choices:              # last chunk = usage only
            continue
        yield chunk.choices[0].delta.content or ""

if prompt := st.chat_input("Ask about NRP..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)

    with st.chat_message("assistant"):
        answer = st.write_stream(token_stream(st.session_state.messages))
    st.session_state.messages.append({"role": "assistant", "content": answer})

### B4. A sidebar — pick a model, set the temperature, clear the chat

Now add controls and **handle the reasoning-model gotcha** (A2): if a model only emits `reasoning` at first, show a *"🧠 thinking…"* placeholder so the screen is never dead.

In [ ]:
%%writefile app.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"))

st.set_page_config(page_title="NRP Helper", page_icon="🤖")
st.title("🤖 NRP Helper")

with st.sidebar:
    st.header("Settings")
    model = st.selectbox("Model", ["gemma", "gpt-oss", "qwen3-small"])
    temperature = st.slider("Temperature", 0.0, 1.5, 0.7)
    if st.button("🗑️ Clear chat"):
        st.session_state.pop("messages", None)
        st.rerun()

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "system", "content": "You are a helpful, concise NRP support assistant."},
    ]

for msg in st.session_state.messages:
    if msg["role"] != "system":
        st.chat_message(msg["role"]).write(msg["content"])

def token_stream(messages, model, temperature):
    stream = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, stream=True,
    )
    thinking = True
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        if delta.content:
            thinking = False
            yield delta.content
        elif getattr(delta, "reasoning", None) and thinking:
            yield "🧠 thinking…\n\n"       # shown once while the model reasons
            thinking = False

if prompt := st.chat_input("Ask about NRP..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    with st.chat_message("assistant"):
        answer = st.write_stream(token_stream(st.session_state.messages, model, temperature))
    # strip the placeholder before saving to history
    answer = answer.replace("🧠 thinking…\n\n", "")
    st.session_state.messages.append({"role": "assistant", "content": answer})

---
## Part C — Tools show-and-tell

An LLM can't multiply large numbers reliably and it *definitely* can't see your cluster. **Tool calling** fixes both: you hand the model a menu of functions; it decides when to call one; you run it and hand back the result.

The loop is always the same four steps:

1. Send the messages **+ a `tools` list**.
2. If the reply has `tool_calls`, the model wants a function run.
3. **You** run the function and append a `{"role": "tool", ...}` message with the result.
4. Call the model again — now it answers using the result.

### C1. The simplest tool — exact math

In [ ]:
def multiply(a, b):
    return a * b

tools = [{
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers exactly.",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
    },
}]

messages = [{"role": "user", "content": "What is 2347 times 891?"}]
resp = client.chat.completions.create(model="gpt-oss", messages=messages, tools=tools)
call = resp.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)
print("model wants:", call.function.name, args)

result = multiply(**args)
messages.append(resp.choices[0].message)                       # the assistant's tool request
messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
final = client.chat.completions.create(model="gpt-oss", messages=messages, tools=tools)
print("bot:", final.choices[0].message.content)

### C2. A reusable tool loop

Real bots may call several tools before answering, so wrap the four steps in a loop. Give it a **registry** mapping tool name → Python function, and it handles any tool you add.

In [ ]:
def chat_with_tools(user_text, tools, registry, model="gpt-oss", system=None, max_rounds=6):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_text})
    for _ in range(max_rounds):
        resp = client.chat.completions.create(model=model, messages=messages, tools=tools)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            fn = registry[call.function.name]
            args = json.loads(call.function.arguments)
            result = fn(**args)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
    return "(stopped after too many tool rounds)"

print(chat_with_tools("What is 2347 * 891, and is the result even?",
                      tools, {"multiply": multiply}))

### C3. A **real** tool — read-only `kubectl`

Now the fun one: let the bot inspect your actual cluster. **Safety first** — we only ever let the model run *read-only* verbs (`get`, `describe`, `logs`, `top`) in *our* namespace. Never let an LLM run `delete`, `apply`, or `exec`. The allowlist is the whole point.

In [ ]:
ALLOWED = {"get", "describe", "logs", "top"}

def run_kubectl(args):
    # args is a kubectl subcommand string, e.g. "get pods" or "get deployments".
    parts = args.split()
    if not parts or parts[0] not in ALLOWED:
        return f"Refused: only read-only verbs {sorted(ALLOWED)} are allowed."
    cmd = ["kubectl", *parts, "-n", NAMESPACE]
    out = subprocess.run(cmd, capture_output=True, text=True, timeout=20)
    return (out.stdout or out.stderr or "(no output)")[:3000]

kube_tools = [{
    "type": "function",
    "function": {
        "name": "run_kubectl",
        "description": "Run a READ-ONLY kubectl command (get/describe/logs/top) in the REHS namespace and return its output.",
        "parameters": {
            "type": "object",
            "properties": {"args": {"type": "string",
                "description": "kubectl subcommand, e.g. 'get pods' or 'get deployments'"}},
            "required": ["args"],
        },
    },
}]

sys_prompt = (f"You are an NRP cluster assistant. The namespace is {NAMESPACE}. "
              "Use run_kubectl to check REAL cluster state before you answer. Be concise.")

print(chat_with_tools("How many pods are running in our namespace, and what are they?",
                      kube_tools, {"run_kubectl": run_kubectl}, system=sys_prompt))

Try more questions — the same loop handles them because the model picks the `kubectl` command itself:

- *"Are any pods not Ready or crashing?"*
- *"What deployments exist and how many replicas does each want?"*
- *"Show me the recent events in the namespace."*

**🔧 Your turn:** add a **second** tool — a plain `get_current_time()` (no arguments) — to the registry and tools list, then ask *"what time is it, and how many pods are running?"* Watch the bot call **both**.

In [ ]:
# 🔧 Your turn — fill this in.
from datetime import datetime

def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# TODO: add a tool schema for get_current_time to a new `tools2` list (copy kube_tools[0] as a template),
# TODO: build a registry {"run_kubectl": run_kubectl, "get_current_time": get_current_time},
# TODO: call chat_with_tools("what time is it, and how many pods are running?", tools2, registry2, system=sys_prompt)

---
## Part D — Put the kubectl tool in the web app

Finally, fold tool calling into Streamlit. This app's bot can **look at the cluster** and answer in the browser. (We render the final answer non-streamed to keep the tool loop simple — a fine trade for a show-and-tell.)

In [ ]:
%%writefile app.py
import os, json, subprocess
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"))

NAMESPACE = "rehs-2026-chatbot"          # your cohort's shared namespace
ALLOWED = {"get", "describe", "logs", "top"}

def run_kubectl(args):
    parts = args.split()
    if not parts or parts[0] not in ALLOWED:
        return f"Refused: only read-only verbs {sorted(ALLOWED)} are allowed."
    out = subprocess.run(["kubectl", *parts, "-n", NAMESPACE],
                         capture_output=True, text=True, timeout=20)
    return (out.stdout or out.stderr or "(no output)")[:3000]

TOOLS = [{"type": "function", "function": {
    "name": "run_kubectl",
    "description": "Run a READ-ONLY kubectl command (get/describe/logs/top) in the namespace.",
    "parameters": {"type": "object",
        "properties": {"args": {"type": "string", "description": "e.g. 'get pods'"}},
        "required": ["args"]}}}]
REGISTRY = {"run_kubectl": run_kubectl}

def answer(messages, model="gpt-oss"):
    for _ in range(6):
        resp = client.chat.completions.create(model=model, messages=messages, tools=TOOLS)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            result = REGISTRY[call.function.name](**json.loads(call.function.arguments))
            with st.expander(f"🔧 ran: kubectl {json.loads(call.function.arguments)['args']}"):
                st.code(result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "(stopped after too many tool rounds)"

st.set_page_config(page_title="NRP Cluster Bot", page_icon="🤖")
st.title("🤖 NRP Cluster Bot")
st.caption(f"I can inspect the **{NAMESPACE}** namespace (read-only).")

if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "system", "content":
        f"You are an NRP cluster assistant. Namespace is {NAMESPACE}. "
        "Use run_kubectl to check real cluster state before answering. Be concise."}]

for msg in st.session_state.messages:
    if msg["role"] not in ("system", "tool") and isinstance(msg.get("content"), str):
        st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("e.g. how many pods are running?"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    with st.chat_message("assistant"):
        reply = answer(st.session_state.messages)
        st.write(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})

Run it and ask:

```bash
streamlit run app.py
```

- *"How many pods are running in our namespace?"*
- *"Is anything crashing?"*
- *"What deployments exist here?"*

You'll see a **🔧 ran: kubectl …** expander showing the exact command the bot chose, then its answer. That's your Week 8 preview: a bot that helps you use NRP, **running on and reasoning about NRP itself.**

---
## Part E — Ship the code *onto* the cluster with a ConfigMap

So far you run `app.py` on **your laptop**. To get it running as a pod on NRP, the cluster needs your code. The heavyweight way is to build a Docker image and push it to a registry. But for a small app there's a lighter trick your teammates already use: **put `app.py` in a ConfigMap** and have a Deployment **mount it as a file**. No image build, no registry — your code ships as cluster *data*.

The moving parts:

- **ConfigMap** `nrp-helper-app` — holds the text of `app.py`.
- **Secret** `nrp-llm-token` — holds your NRP token (a Secret, **never** a ConfigMap, **never** git).
- **Deployment** — runs a stock `python:3.12-slim` image, `pip install`s the deps at startup, **mounts the ConfigMap at `/app`**, and runs `streamlit run /app/app.py`. The token arrives as an env var from the Secret.

> This is the same shape as the deployment you saw in Week 3 — the new idea is the `volume` + `volumeMounts` that turn a ConfigMap into a file inside the container.

### E1. Put `app.py` into a ConfigMap (and the token into a Secret)

Use the **B3 streaming app** from Part B for this — a self-contained chat UI with no `kubectl` (a pod has no kubeconfig, so the Part D tool app wouldn't work here). Re-run the B3 cell so a fresh `app.py` is on disk, then:

```bash
# code -> ConfigMap (re-run this whenever you change app.py)
kubectl create configmap nrp-helper-app --from-file=app.py \
  -n rehs-2026-chatbot --dry-run=client -o yaml | kubectl apply -f -

# token -> Secret (never put this in the ConfigMap or in git)
kubectl create secret generic nrp-llm-token \
  --from-literal=NRP_LLM_TOKEN=sk-...your token... \
  -n rehs-2026-chatbot --dry-run=client -o yaml | kubectl apply -f -
```

The `--dry-run=client -o yaml | kubectl apply -f -` pattern is an **upsert**: it creates the object the first time and updates it on every re-run, instead of erroring with "already exists".

In [ ]:
# Write app.py to disk from Python (same content as B3), so E1 has something to package.
b3_app = '''import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"))

st.set_page_config(page_title="NRP Helper", page_icon="\U0001f916")
st.title("\U0001f916 NRP Helper — running on NRP")

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "system", "content": "You are a helpful, concise NRP support assistant."},
    ]

for msg in st.session_state.messages:
    if msg["role"] != "system":
        st.chat_message(msg["role"]).write(msg["content"])

def token_stream(messages):
    stream = client.chat.completions.create(model="gemma", messages=messages, stream=True)
    for chunk in stream:
        if not chunk.choices:
            continue
        yield chunk.choices[0].delta.content or ""

if prompt := st.chat_input("Ask about NRP..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    with st.chat_message("assistant"):
        answer = st.write_stream(token_stream(st.session_state.messages))
    st.session_state.messages.append({"role": "assistant", "content": answer})
'''
with open("app.py", "w") as f:
    f.write(b3_app)

# Package the code into a ConfigMap (upsert). Run this whenever app.py changes.
print(subprocess.run(
    "kubectl create configmap nrp-helper-app --from-file=app.py "
    f"-n {NAMESPACE} --dry-run=client -o yaml | kubectl apply -f -",
    shell=True, capture_output=True, text=True).stdout)

# The token goes in a Secret, not the ConfigMap. Reads NRP_LLM_TOKEN from your env/.env.
print(subprocess.run(
    "kubectl create secret generic nrp-llm-token "
    f"--from-literal=NRP_LLM_TOKEN={os.environ['NRP_LLM_TOKEN']} "
    f"-n {NAMESPACE} --dry-run=client -o yaml | kubectl apply -f -",
    shell=True, capture_output=True, text=True).stdout)

### E2. A Deployment that mounts the ConfigMap and runs Streamlit

Read this manifest top-to-bottom — the important lines are commented:

- `command` / `args` — install the deps into the stock image, then start Streamlit bound to `0.0.0.0` so the cluster can reach it.
- `env` → `secretKeyRef` — pull the token out of the Secret into `NRP_LLM_TOKEN`.
- `volumes` → `configMap` and `volumeMounts` → `/app` — this is the trick: the ConfigMap's `app.py` key becomes the file `/app/app.py` inside the container.

The `Service` gives the pod a stable in-cluster name so you (and, in Week 7, an Ingress) can reach it.

In [ ]:
%%writefile deploy.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nrp-helper
  namespace: rehs-2026-chatbot
  labels: {app: nrp-helper}
spec:
  replicas: 1
  selector:
    matchLabels: {app: nrp-helper}
  template:
    metadata:
      labels: {app: nrp-helper}
    spec:
      containers:
        - name: streamlit
          image: python:3.12-slim              # stock image, no build needed
          command: ["/bin/sh", "-c"]
          args:
            - >
              pip install --quiet --no-cache-dir streamlit openai python-dotenv &&
              streamlit run /app/app.py
              --server.port=8501 --server.address=0.0.0.0 --server.headless=true
          env:
            - name: NRP_LLM_TOKEN               # <- injected from the Secret
              valueFrom:
                secretKeyRef: {name: nrp-llm-token, key: NRP_LLM_TOKEN}
            - name: NRP_LLM_BASE_URL
              value: https://ellm.nrp-nautilus.io/v1
          ports:
            - {containerPort: 8501}
          readinessProbe:                       # not "Ready" until Streamlit accepts connections
            tcpSocket: {port: 8501}
            initialDelaySeconds: 20
            periodSeconds: 5
            failureThreshold: 30                 # allow ~2.5 min for the first pip install
          volumeMounts:
            - {name: app-code, mountPath: /app} # <- ConfigMap shows up here as files
          resources:
            requests: {cpu: "250m", memory: "512Mi"}
            limits:   {cpu: "1",    memory: "1Gi"}
      volumes:
        - name: app-code
          configMap: {name: nrp-helper-app}     # <- the ConfigMap from E1
---
apiVersion: v1
kind: Service
metadata:
  name: nrp-helper
  namespace: rehs-2026-chatbot
spec:
  selector: {app: nrp-helper}
  ports:
    - {port: 8501, targetPort: 8501}

### E3. Deploy it, watch it come up, and reach it

```bash
kubectl apply -f deploy.yaml
kubectl rollout status deploy/nrp-helper -n rehs-2026-chatbot   # waits until Ready
kubectl logs -l app=nrp-helper -n rehs-2026-chatbot --tail=5    # look for "Uvicorn server started"
```

The pod spends its first ~30–60s running `pip install`, so it isn't reachable instantly. The `readinessProbe` in the manifest keeps the pod out of "Ready" until Streamlit is actually accepting connections on 8501, so `rollout status` genuinely blocks until the app is up (not just until the container process starts). Then port-forward the Service to your laptop and open it:

```bash
kubectl port-forward -n rehs-2026-chatbot svc/nrp-helper 8501:8501
# now open http://localhost:8501  ->  your app, running as a pod on NRP
```

In [ ]:
# Apply, then wait for the rollout (pip install makes the first boot slow).
print(subprocess.run(["kubectl", "apply", "-f", "deploy.yaml"],
                     capture_output=True, text=True).stdout)
print(subprocess.run(["kubectl", "rollout", "status", "deploy/nrp-helper",
                      "-n", NAMESPACE, "--timeout=180s"],
                     capture_output=True, text=True).stdout)

# Confirm the pod is Running and Streamlit booted.
print(subprocess.run(["kubectl", "get", "pods", "-l", "app=nrp-helper",
                      "-n", NAMESPACE], capture_output=True, text=True).stdout)
print(subprocess.run(["kubectl", "logs", "-l", "app=nrp-helper",
                      "-n", NAMESPACE, "--tail=4"], capture_output=True, text=True).stdout)

**The mental model:** the ConfigMap *is* your code, living in the cluster. Change `app.py`, re-run the E1 ConfigMap command to push the new version, then `kubectl rollout restart deploy/nrp-helper` so a fresh pod picks it up. That edit-push-restart loop is exactly how you'll iterate on the real bot in Weeks 7–8.

**When to graduate to a Docker image:** ConfigMaps are capped at ~1 MiB and only carry your own source — fine for one `app.py`, painful once you have many files or pip installs that should be baked in (not re-run on every boot). Week 7 makes that jump. For now, this gets your bot onto NRP today.

Clean up when you're done experimenting:

```bash
kubectl delete deploy/nrp-helper svc/nrp-helper -n rehs-2026-chatbot
kubectl delete configmap/nrp-helper-app secret/nrp-llm-token -n rehs-2026-chatbot
```

---
## Where this is going + challenges

You now have every ingredient of the final product except **retrieval** (Week 5): a web UI, streaming, memory, personas, and tools.

**Challenges**
- 🧮 Add the `multiply` tool *and* `run_kubectl` to the Part D app so the bot has two hands.
- 🧠 Stream the final answer in Part D too (hint: after the tool loop finishes, make one last `stream=True` call).
- 🔒 Harden `run_kubectl`: reject `-o jsonpath` tricks, cap output, add a timeout message. Why is an allowlist safer than a blocklist?
- 🎭 Give the bot a persona from a `system.txt` file the sidebar can edit live.
- 📊 Add a token counter to the sidebar (`response.usage.total_tokens`).

**Never** wire a write/delete verb into an LLM tool on a shared cluster. The allowlist is not optional.